In [25]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [26]:
import pandas as pd

print(pd.__version__)


3.0.6


# 1. Look

In [27]:
import pandas as pd

file_path = "yellow_tripdata_2021-01.csv.gz"

df = pd.read_csv(file_path, nrows=100)

print(df.head())
print(df.dtypes)

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21                1   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10           1                  N           142            43   
1           0.20           1                  N           238           151   
2          14.70           1                  N           132           165   
3          10.60           1                  N           138           132   
4           4.94           1                  N            68            33   

   payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  \


In [28]:
print(df.shape)
print(df.dtypes)

(100, 18)
VendorID                   int64
tpep_pickup_datetime         str
tpep_dropoff_datetime        str
passenger_count            int64
trip_distance            float64
RatecodeID                 int64
store_and_fwd_flag           str
PULocationID               int64
DOLocationID               int64
payment_type               int64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
improvement_surcharge    float64
total_amount             float64
congestion_surcharge     float64
dtype: object


# 2. Profile

In [29]:

import pandas as pd

file_path = "yellow_tripdata_2021-01.csv.gz"

chunk_size = 100_000

total_rows = 0


In [30]:


total_rows = 0

null_counts = {}
min_values = {}
max_values = {}
dtypes = {}

for chunk in pd.read_csv(file_path, chunksize=chunk_size):

    total_rows += len(chunk)

    for col in chunk.columns:
        null_counts[col] = (
            null_counts.get(col, 0)
            + chunk[col].isna().sum()
        )

        dtypes[col] = str(chunk[col].dtype)

        col_min = chunk[col].min()
        col_max = chunk[col].max()

        if pd.notna(col_min):
            if col not in min_values or col_min < min_values[col]:
                min_values[col] = col_min

        if pd.notna(col_max):
            if col not in max_values or col_max > max_values[col]:
                max_values[col] = col_max

print("Total rows:", total_rows)


Total rows: 1369765


C:\Users\101\AppData\Local\Temp\ipykernel_2636\3625642816.py:8: DtypeWarning: Columns (6: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file_path, chunksize=chunk_size):


In [31]:

profile_df = pd.DataFrame({
    "dtype": dtypes,
    "nulls": null_counts,
    "null_%": {
        col: round(null_counts[col] / total_rows * 100, 2)
        for col in null_counts
    },
    "min": min_values,
    "max": max_values
})

print(profile_df)


                         dtype  nulls  null_%                  min  \
VendorID               float64  98352    7.18                    1   
tpep_pickup_datetime       str      0    0.00  2008-12-31 23:05:14   
tpep_dropoff_datetime      str      0    0.00  2008-12-31 23:07:22   
passenger_count        float64  98352    7.18                    0   
trip_distance          float64      0    0.00                  0.0   
RatecodeID             float64  98352    7.18                    1   
store_and_fwd_flag     float64  98352    7.18                    N   
PULocationID             int64      0    0.00                    1   
DOLocationID             int64      0    0.00                    1   
payment_type           float64  98352    7.18                    1   
fare_amount            float64      0    0.00               -490.0   
extra                  float64      0    0.00                 -5.5   
mta_tax                float64      0    0.00                 -0.5   
tip_amount          

# 3. Find problems

In [32]:
#P1

date_issues = 0

for chunk in pd.read_csv(file_path, chunksize=chunk_size):

    pickup = pd.to_datetime(
        chunk["tpep_pickup_datetime"],
        errors="coerce"
    )

    outside_january = (
        (pickup < "2021-01-01") |
        (pickup >= "2021-02-01") |
        (pickup.isna())
    )

    date_issues += outside_january.sum()

print("Pickup dates outside January or invalid:", date_issues)


# Decision : Drop , Reason : The dataset is intended to cover January 2021. Records with invalid pickup dates or dates outside the required period do not meet the dataset's scope.




Pickup dates outside January or invalid: 24


C:\Users\101\AppData\Local\Temp\ipykernel_2636\2138920280.py:5: DtypeWarning: Columns (6: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file_path, chunksize=chunk_size):


In [33]:
#p2:

invalid_fares = 0

for chunk in pd.read_csv(file_path, chunksize=chunk_size):

    problem = chunk["fare_amount"] <= 0

    invalid_fares += problem.sum()

print("Zero or negative fares:", invalid_fares)


# Decision : Flag for Review , Reason : Negative or zero fares may indicate refunds, chargebacks, voided trips, or other unusual transactions. Instead of dropping these records automatically, we flag them for review because their exact meaning cannot be confirmed from the fare amount alone. This preserves potentially useful data while allowing the data quality or finance team to investigate. The raw file remains untouched.



Zero or negative fares: 7411


C:\Users\101\AppData\Local\Temp\ipykernel_2636\1724788923.py:5: DtypeWarning: Columns (6: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file_path, chunksize=chunk_size):


In [34]:
#P3

invalid_distances = 0

for chunk in pd.read_csv(file_path, chunksize=chunk_size):

    problem = chunk["trip_distance"] <= 0

    invalid_distances += problem.sum()

print("Zero or negative trip distances:", invalid_distances)


# Decision : Drop , Reason : A trip with no positive recorded distance is not suitable for distance-based trip analysis.




Zero or negative trip distances: 19952


C:\Users\101\AppData\Local\Temp\ipykernel_2636\4175944532.py:5: DtypeWarning: Columns (6: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file_path, chunksize=chunk_size):


In [35]:
#P4

zero_passengers = 0

for chunk in pd.read_csv(file_path, chunksize=chunk_size):

    problem = chunk["passenger_count"] == 0

    zero_passengers += problem.sum()

print("Trips with zero passengers:", zero_passengers)


# Decision : Drop , Reason : A passenger count of zero is not valid for a standard passenger taxi trip and may indicate incorrect or incomplete data.

Trips with zero passengers: 26726


C:\Users\101\AppData\Local\Temp\ipykernel_2636\3167983343.py:5: DtypeWarning: Columns (6: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file_path, chunksize=chunk_size):


In [ ]:
#P5

import hashlib

seen_rows = set()
duplicate_rows = 0

for chunk in pd.read_csv(file_path, chunksize=chunk_size):

    row_hashes = pd.util.hash_pandas_object(
        chunk,
        index=False
    )

    for row_hash in row_hashes:
        if row_hash in seen_rows:
            duplicate_rows += 1
        else:
            seen_rows.add(row_hash)

print("Duplicate rows:", duplicate_rows)


# Decision : No Action , Reason : No duplicate rows were detected, so no duplicate-removal rule is needed based on the current check.

C:\Users\101\AppData\Local\Temp\ipykernel_2636\2305866138.py:8: DtypeWarning: Columns (6: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file_path, chunksize=chunk_size):


Duplicate rows: 0


# 6. Clean

In [37]:

def clean(df):
    cleaned_df = df.copy()

    rows_removed = {
        "invalid_pickup_dates": 0,
        "invalid_trip_distances": 0,
        "zero_passengers": 0
    }

    # Rule 1: Remove invalid pickup dates or dates outside January 2021
    pickup_dates = pd.to_datetime(
        cleaned_df["tpep_pickup_datetime"],
        errors="coerce"
    )

    invalid_dates = (
        (pickup_dates < "2021-01-01") |
        (pickup_dates >= "2021-02-01") |
        pickup_dates.isna()
    )

    rows_removed["invalid_pickup_dates"] = int(invalid_dates.sum())
    cleaned_df = cleaned_df.loc[~invalid_dates].copy()

    # Rule 2: Remove zero or negative trip distances
    invalid_distances = cleaned_df["trip_distance"] <= 0

    rows_removed["invalid_trip_distances"] = int(
        invalid_distances.sum()
    )
    cleaned_df = cleaned_df.loc[~invalid_distances].copy()

    # Rule 3: Remove trips with zero passengers
    zero_passengers = cleaned_df["passenger_count"] == 0

    rows_removed["zero_passengers"] = int(
        zero_passengers.sum()
    )
    cleaned_df = cleaned_df.loc[~zero_passengers].copy()

    # Rule 4: Flag missing, zero, or negative fares for review
    invalid_fares = (
        cleaned_df["fare_amount"].isna() |
        (cleaned_df["fare_amount"] <= 0)
    )

    cleaned_df["fare_review_flag"] = invalid_fares

    return cleaned_df, rows_removed


# Test First 100 Row

In [38]:

df_test = pd.read_csv(file_path, nrows=100)

cleaned_test, removed_test = clean(df_test)

print("Rows before cleaning:", len(df_test))
print("Rows after cleaning:", len(cleaned_test))
print("Rows removed by rule:", removed_test)
print("Original DataFrame rows:", len(df_test))


Rows before cleaning: 100
Rows after cleaning: 95
Rows removed by rule: {'invalid_pickup_dates': 0, 'invalid_trip_distances': 2, 'zero_passengers': 3}
Original DataFrame rows: 100


In [39]:

total_before = 0
total_after = 0
total_fares_flagged = 0

total_removed = {
    "invalid_pickup_dates": 0,
    "invalid_trip_distances": 0,
    "zero_passengers": 0
}

for chunk in pd.read_csv(file_path, chunksize=chunk_size):

    total_before += len(chunk)

    cleaned_chunk, removed = clean(chunk)

    total_after += len(cleaned_chunk)

    total_fares_flagged += int(
        cleaned_chunk["fare_review_flag"].sum()
    )

    for rule in total_removed:
        total_removed[rule] += removed[rule]

print("Total rows before cleaning:", total_before)
print("Total rows after cleaning:", total_after)
print("Total rows removed:", total_before - total_after)
print("Rows removed by rule:", total_removed)
print("Rows flagged for fare review:", total_fares_flagged)


C:\Users\101\AppData\Local\Temp\ipykernel_2636\4276454408.py:11: DtypeWarning: Columns (6: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file_path, chunksize=chunk_size):


Total rows before cleaning: 1369765
Total rows after cleaning: 1323532
Total rows removed: 46233
Rows removed by rule: {'invalid_pickup_dates': 24, 'invalid_trip_distances': 19951, 'zero_passengers': 26258}
Rows flagged for fare review: 6415


# Clean The Missing Values

In [40]:

# Check missing values before removal
df_full = pd.read_csv(file_path)

print("Rows before removing missing values:", len(df_full))

print("\nMissing values before removal:")
print(df_full.isnull().sum())

# Remove rows containing any missing values
df_no_missing = df_full.dropna().copy()

print("\nRows after removing missing values:", len(df_no_missing))
print("Rows removed:", len(df_full) - len(df_no_missing))

print("\nMissing values after removal:")
print(df_no_missing.isnull().sum().sum())



C:\Users\101\AppData\Local\Temp\ipykernel_2636\3313220989.py:2: DtypeWarning: Columns (6: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df_full = pd.read_csv(file_path)


Rows before removing missing values: 1369765

Missing values before removal:
VendorID                 98352
tpep_pickup_datetime         0
tpep_dropoff_datetime        0
passenger_count          98352
trip_distance                0
RatecodeID               98352
store_and_fwd_flag       98352
PULocationID                 0
DOLocationID                 0
payment_type             98352
fare_amount                  0
extra                        0
mta_tax                      0
tip_amount                   0
tolls_amount                 0
improvement_surcharge        0
total_amount                 0
congestion_surcharge         0
dtype: int64

Rows after removing missing values: 1271413
Rows removed: 98352

Missing values after removal:
0


In [41]:
print("Missing values after removal:")
print(df_no_missing.isnull().sum())

Missing values after removal:
VendorID                 0
tpep_pickup_datetime     0
tpep_dropoff_datetime    0
passenger_count          0
trip_distance            0
RatecodeID               0
store_and_fwd_flag       0
PULocationID             0
DOLocationID             0
payment_type             0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    0
total_amount             0
congestion_surcharge     0
dtype: int64
